# Summary statistics: center, spread, shape, position

_Data Wrangling_

**A few numbers that stand in for a whole column — and which ones to trust when the data is skewed.**

A column can have thousands of values, but a person can only hold a few numbers in their head. A
       summary statistic is a single number that stands in for the whole column along one axis. The
       craft is knowing which axis you are summarizing and which statistic tells the truth for this
       column's shape.

       There are four axes:

---

This notebook builds the summary up one statistic at a time — center, then spread, then shape, then what an outlier does to each. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — pandas + scipy

### Step 1 — Load a real, right-skewed column

We pull one column out of the breast-cancer dataset: `mean area`, the average nucleus area across 569 cell samples. It is **strongly right-skewed** — most values are modest but a long tail of large nuclei stretches the high end. That skew is exactly what makes the choice of statistic matter.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import median_abs_deviation
from sklearn.datasets import load_breast_cancer

# A real, right-skewed feature: cell-nucleus 'mean area'.
df = load_breast_cancer(as_frame=True).frame
s = df['mean area']                      # 569 values, strongly right-skewed

### Step 2 — The describe() reflex

`Series.describe()` is the first thing most people reach for: it returns count, mean, std, min, the three quartiles, and max in one shot. Read it carefully here — the mean (654.89) sits **above** the median (the 50% row, 551.10), and that gap is the tell-tale fingerprint of a right skew.

In [ ]:
# count, mean, std, min, quartiles, max — all at once.
print(s.describe())
# count    569.00
# mean     654.89   <- the average
# std      351.91   <- spread around the mean (non-robust)
# min      143.50
# 25%      420.30   <- Q1
# 50%      551.10   <- median  (mean 654.89 > median 551.10  => right skew)
# 75%      782.70   <- Q3
# max     2501.00

### Step 3 — The robust + shape stats describe() leaves out

`describe()` reports the mean and std, which every value pulls on (non-robust). The **robust** center and spread — median, IQR (`Q3 − Q1`), and MAD (median absolute deviation) — lean on ranks and middles, so the long tail can't drag them around. We also add **skew** and **kurtosis**, the two numbers that describe the column's *shape* rather than its center or spread.

In [ ]:
# Robust center and two robust spreads.
median = s.median()                                  # 551.10  robust center
iqr    = s.quantile(0.75) - s.quantile(0.25)         # 362.40  robust spread
mad    = median_abs_deviation(s)                     # 153.30  robust spread (scipy)
print('median:', round(median, 2), ' IQR:', round(iqr, 2), ' MAD:', round(mad, 2))

# Shape: skew (lean) and kurtosis (tail weight).
print('skew:', round(s.skew(), 3), ' kurtosis:', round(s.kurt(), 3))
# skew 1.646 (long right tail), kurtosis 3.652 (heavier-than-normal tails)

# Five-number summary + any percentiles you like:
five_number = s.quantile([0, 0.25, 0.5, 0.75, 1.0]).round(1).tolist()
print(five_number)                                   # min, Q1, med, Q3, max

deciles = s.quantile([0.10, 0.90]).round(1).tolist()
print(deciles)                                       # 10th / 90th pctile

### Step 4 — Watch one outlier move the mean but not the median

Here is the punchline made concrete. We take a clean little column, then add a single absurd value (1000) and recompute. The **mean** lurches from 11.80 to 101.64; the **median** does not move at all. That is why, when mean and median disagree, you report the median and IQR and reserve mean and std for symmetric, clean columns.

In [ ]:
# A clean column: mean and median agree.
clean = pd.Series([10, 12, 11, 13, 12, 10, 14, 11, 12, 13.])
print('clean  -> mean %.2f  median %.2f' % (clean.mean(), clean.median()))
# clean  -> mean 11.80  median 12.00

# Add one wild outlier and recompute.
dirty = pd.concat([clean, pd.Series([1000.])], ignore_index=True)
print('dirty  -> mean %.2f  median %.2f' % (dirty.mean(), dirty.median()))
# dirty  -> mean 101.64 median 12.00   <- mean ruined, median unmoved

# Rule of thumb: when mean and median disagree (skew is large), report the
# median + IQR (robust); reserve mean + std for symmetric, clean columns.

## Visualize the data & results

_For a strongly right-skewed real feature ('mean area' in load_breast_cancer), how far apart are the robust and non-robust summaries — does the mean really exceed the median, and is the std really inflated relative to the IQR?_

### Step 1 — Recompute every summary side by side

We reload the same right-skewed `mean area` column and compute the full set of summaries together: the non-robust pair (mean, std) and the robust ones (median, IQR, MAD). Lining them up is what lets us measure how far apart the two families land on a skewed column.

In [ ]:
import numpy as np
from scipy.stats import median_abs_deviation
from sklearn.datasets import load_breast_cancer

df = load_breast_cancer(as_frame=True).frame
s = df['mean area']                       # 569 real values, strongly right-skewed

mean   = s.mean()                         # 654.89
median = s.median()                       # 551.10
std    = s.std()                          # 351.91
iqr    = s.quantile(0.75) - s.quantile(0.25)   # 362.40
mad    = median_abs_deviation(s)          # 153.30

### Step 2 — Compare robust vs non-robust

Now read the gaps. The skew is 1.646 — a clear right lean. The mean is pulled well above the median, and the std (351.91) is inflated relative to the MAD (153.30) because squaring the long-tail distances blows it up. The IQR happens to sit close to the std here, but it is the MAD that shows how tight the bulk of the data really is.

In [ ]:
print('skewness :', round(s.skew(), 3))   # 1.646  (long right tail)
print('center   -> median %.2f  mean %.2f' % (median, mean))
print('spread   -> MAD %.2f  IQR %.2f  std %.2f' % (mad, iqr, std))
# center  -> median 551.10  mean 654.89   (mean pulled above median)
# spread  -> MAD 153.30  IQR 362.40  std 351.91   (std inflated by the tail)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your df.describe() for a session_minutes column shows mean 42, std 95, min 0, 25% 3, 50% 9, 75% 31, max 1440. A teammate writes "users spend about 42 minutes per session." What is wrong, and what should the report say?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the mean (42) to the median (the 50% row, 9). — _The mean is more than four times the median, the signature of a strong right skew or outliers._
- Look at the std (95) versus the IQR ($Q_3-Q_1 = 31-3 = 28$) and the max (1440). — _The std is larger than the IQR and the max dwarfs $Q_3$ — a few enormous sessions are inflating the mean and the std together._
- Report the median and IQR instead: "half of sessions are under 9 minutes; the middle 50% span 3 to 31 minutes." — _Robust summaries describe the typical user honestly; the mean describes a user pulled up by rare marathon sessions._

**Answer:** The column is heavily right-skewed (mean 42 versus median 9, max 1440), so the mean overstates the typical session. Report the median (9 minutes) and the IQR (Inter-Quartile Range) of $31-3 = 28$ minutes (the middle 50% from 3 to 31). The std of 95 is inflated by the same long tail and should not be paired with this skewed mean.

</details>

**Problem 2.** You add a single fat-fingered data-entry error of 9999 to a clean column of prices that range 5 to 80. Which of mean, median, std, IQR, and MAD change a lot, and which barely move? Why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Classify each statistic as robust or non-robust. — _Mean and std sum/square every value (non-robust); median, IQR, and MAD are built from middles and ranks (robust)._
- Trace the outlier through the mean and std. — _The mean rises by $9999/n$ and the std, which squares the huge distance $(9999-\bar{x})$, explodes — both move a lot._
- Trace it through the median, IQR, and MAD. — _One point added on the far right shifts the middle of the sorted data by at most one position, so the median, the quartiles ($Q_1,Q_3$), and the MAD barely budge._

**Answer:** The mean and std change a lot — the mean climbs by $9999/n$ and the std blows up because it squares the outlier's distance. The median, IQR, and MAD (Median Absolute Deviation) barely move, because they are computed from ranks and the middle of the data, where one far-out point has no leverage. This is exactly why you reach for the robust trio on dirty data.

</details>

**Problem 3.** Two columns, A and B, both have mean 50 and std 10. Can you conclude they have the same distribution? What two summaries would you add to tell them apart?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that mean and std fix only the center and the spread, not the shape. — _Many different distributions share a mean and a std — symmetric, skewed, bimodal, heavy-tailed._
- Add skewness to each column. — _Skewness reveals whether one column leans (a long tail on one side) while the other is symmetric._
- Add kurtosis (and ideally the five-number summary or a box plot). — _Kurtosis reveals heavy tails / outlier-proneness; the five-number summary shows where the mass actually sits._

**Answer:** No — a shared mean and std fix only center and spread, not shape. One column could be symmetric and the other strongly skewed or bimodal. Add skewness (the lean) and kurtosis (the tail weight), and ideally the five-number summary or a box plot, before declaring the two columns alike.

</details>